In [7]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [8]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [14]:
# creating the backend for simulators
backend = BasicSimulator()

# the random function generates a random bit
# the bit has a 1/3 probability of 0 and a 2/3 probability of 1
from math import acos, sqrt
theta = 2 * acos(1/sqrt(3))
def random():
    rand_q = QuantumCircuit(1, 1)
    rand_q.ry(theta, 0)
    rand_q.measure([0], [0])
    rand_comp = transpile(rand_q, backend)
    rand_sim = backend.run(rand_comp, shots=1)
    rand_qubit = rand_sim.result()
    rand_qubit = int(list(rand_qubit.get_counts().keys())[0])
    return rand_qubit

# this random function generates a random bit 
# the bit has a 1/2 probability of 0 and 1/2 probability of 1
# needed for the second part of random basis choice
def random_50():
    rand_50 = QuantumCircuit(1, 1)
    rand_50.h(0)
    rand_50.measure([0],[0]) 
    rand_comp50 = transpile(rand_50, backend)
    rand_sim50 = backend.run(rand_comp50, shots=1) 
    rand_qubit50 = rand_sim50.result() 
    rand_qubit50 = int(list(rand_qubit50.get_counts().keys())[0])
    return rand_qubit50

    
# this creates the entangled state 1/sqrt(2)(|01>-|10>)
def entangled_qubit():
    entang = QuantumCircuit(2, 2)
    entang.h(0)
    entang.cx(0, 1)
    entang.x(0)
    entang.z(1)
    return entang

# function to choose alice's operator
def alice_choice():
    Alice_Op = ""
    qubit = random() 
    # we have a 1/3 probability of 0 and a 2/3 probability of 1
    if qubit == 0:
        Alice_Op = "X"
    else:
        # to get a probability of 1/3 for W and Z, call random_50() 
        # so we get 2/3 * 1/2 = 1/3 probability for both W and Z
        qubit = random_50()
        if qubit == 0:
            Alice_Op = "W"
        else:
            Alice_Op = "Z"
    return Alice_Op

# function to choose bob's operator
# logic same as for alice
def bob_choice():
    Bob_Op = ""
    qubit = random()
    if qubit == 0:
        Bob_Op = "W"
    else:
        qubit = random_50()
        if qubit == 0:
            Bob_Op = "Z"
        else:
            Bob_Op = "V"
    return Bob_Op

# gets the entangled bits and called the functions for alice and bobs choices
entangled_qubits = entangled_qubit()
Aoperator = alice_choice()
Boperator = bob_choice()


# Operators for W and V from Lab4B
root2 = math.sqrt(2)
denom1 = math.sqrt(4 + 2*root2)
denom2 = math.sqrt(4 - 2*root2) 
B0_transform_matrix = [ [ -1 / denom1 , (1 + root2) / denom1 ],
                        [  1 / denom2 , (root2 - 1) / denom2 ] ]
W = Operator(B0_transform_matrix) 
B1_transform_matrix = [ [  1 / denom1 , (1 + root2) / denom1 ],
                        [ -1 / denom2 , (root2 - 1) / denom2 ] ]
V = Operator(B1_transform_matrix) 

# measures alice and bobs qubits based on their choices 
def alice_meas(entangled_qubits, Aoperator):
    if Aoperator == "X":
        entangled_qubits.h(0)
    elif Aoperator == "W":
        entangled_qubits.unitary(W, [0])
    elif Aoperator == "Z":
        pass

def bob_meas(entangled_qubits, Boperator):
    if Boperator == "W":
        entangled_qubits.unitary(W, [1])
    elif Boperator == "Z":
        pass
    elif Boperator == "V":
        entangled_qubits.unitary(V, [1])

# for entanglement calcultion 
def alice_state_calc(qubit):
    if qubit == 0:
        return 1
    else:
        return -1
    
def bob_state_calc(qubit):
    if qubit == 0:
        return 1
    else: 
        return -1

# the actual E91 protocol, this is the steps repeated 9n/2 times 
# returns a list of tuples, containing alices operator (X, W, or Z),
# bobs operator (W, Z, or V), then alices and bobs state (both either 
# -1 or 1)
def E91_protocol(n):
    prot_results = []
    for i in range(int(9*n/2)):
        entangled_pair = entangled_qubit()
        # the attacker measures bobs qubit
        entangled_pair.measure(1,1)
        alice_operator = alice_choice()
        bob_operator = bob_choice() 
        alice_meas(entangled_pair, alice_operator)
        bob_meas(entangled_pair, bob_operator)
        entangled_pair.measure(range(2), range(2))
        job = backend.run(entangled_pair, shots=1)
        result = job.result()
        count_pair = result.get_counts(entangled_pair)
        count_pair = list(count_pair.keys())[0] 
        alice_bit = int(count_pair[1])
        bob_bit = int(count_pair[0])
        alice_state =  alice_state_calc(alice_bit) 
        bob_state = bob_state_calc(bob_bit) 
        prot_results.append((alice_operator, bob_operator, alice_state, bob_state))
    return prot_results

# this function makes their shared key
# if they are both measuring in W or Z add alices state (-1 or 1)
# to the list, this gives their shared key
def make_key(x):
    key=[]
    for i in range(len(x)):
        make_alice = x[i][0]
        make_bob = x[i][1]
        # print(make_alice, make_bob)
        if make_alice == "W" and make_bob == "W":
            key.append(x[i][2])
        elif make_alice == "Z" and make_bob == "Z":
            key.append(x[i][2])
    return key

# this tests their entanglement 
# S = |⟨X ⊗ W⟩ − ⟨X ⊗ V ⟩ + ⟨Z ⊗ W⟩ + ⟨Z ⊗ V ⟩|
# using i=X and j=W as exmple
# loop through all the tuples, if alices operator is X and bobs is W
# add (alices state * bobs state) to a list, after looping through sum the
# entries of the list and divide by length, then sub that number into the 
# S formula. Same logic for the other 3 (i=X&j=V, i=Z&j=W, i=Z&j=V)
# if the list is emtpy dont do the calculation (as cant divide by 0)
# the combinations are all zero by default
def entanglement_test(x):
    S = 0
    X_W = []
    X_V = [] 
    Z_W = [] 
    Z_V = []
    XW = 0
    XV = 0
    ZW = 0
    ZV = 0
    for i in range(len(x)):
        make_alice = x[i][0]
        make_bob = x[i][1]
        if make_alice == "X" and make_bob == "W":
            pro = (x[i][2]) * (x[i][3])
            X_W.append(pro)
        elif make_alice == "X" and make_bob == "V":
            pro = (x[i][2]) * (x[i][3])
            X_V.append(pro)
        elif make_alice == "Z" and make_bob == "W":
            pro = (x[i][2]) * (x[i][3])
            Z_W.append(pro)
        elif make_alice == "Z" and make_bob == "V":
            pro = (x[i][2]) * (x[i][3])
            Z_V.append(pro)
    if len(X_W) != 0:
        XW = sum(X_W)/len(X_W)
    if len(X_V) != 0:    
        XV = sum(X_V)/len(X_V)
    if len(Z_W) != 0:
        ZW = sum(Z_W)/len(Z_W)
    if len(Z_V) != 0:
        ZV = sum(Z_V)/len(Z_V)
    S = abs(XW - XV + ZW + ZV)
    return S

resultsE91 = E91_protocol(100)
key_test = make_key(resultsE91)
print("The shared key is:", key_test)
S_test = entanglement_test(resultsE91)
print("S:", S_test)

The shared key is: [1, 1, 1, -1, -1, -1, 1, -1, 1, -1, 1, -1, 1, 1, 1, -1, 1, 1, 1, -1, -1, -1, 1, 1, -1, 1, -1, -1, 1, -1, -1, 1, -1, -1, 1, 1, -1, -1, -1, -1, 1, -1, 1, 1, -1, 1, 1, -1, -1, -1, -1, -1, -1, 1, -1, -1, -1, 1, -1, 1, 1, 1, -1, 1, -1, -1, -1, 1, 1, 1, -1, 1, -1, 1, 1, 1, 1, -1, -1, 1, 1, 1, -1, -1, 1, 1, 1, -1, -1, -1, 1, -1, 1, -1, -1, -1, 1, -1, 1, 1, 1, -1]
S: 1.2118214011901718
